# LoRA × SegFormer-B0 — Semantic Segmentation on SceneParse150

This notebook extends the repo's from-scratch LoRA implementation to a **semantic segmentation** model. The original Microsoft LoRA paper did **not** report segmentation experiments; this is a practical vision extension built on top of the same low-rank `ΔW = BA` idea.

**Setup in this notebook**
- Backbone: `nvidia/mit-b0` loaded with `AutoModelForSemanticSegmentation`
- Dataset: `merve/scene_parse_150` (the public ADE20K subset used in the Hugging Face semantic segmentation guide)
- LoRA targets: attention `query` and `value` projections inside the SegFormer encoder
- Trainable params: LoRA adapters + `decode_head`
- Metrics: mean IoU and pixel accuracy

**Runtime budget (Colab T4)**
- Smoke test (`train[:200]`, 256px, 6 epochs): roughly 20–40 minutes
- Better run: increase the dataset slice or switch `dataset_split` to `train`


In [ ]:
!nvidia-smi


## 1. Load the project code

Pick **one** option below.

**Option A (recommended):** zip the repo's `code/` folder and upload it here.
```bash
# on your laptop, from inside project_LoRA/:
zip -r code.zip code -x 'code/external/*' -x '*/__pycache__/*' -x '*/.venv/*'
```

**Option B:** clone the public GitHub mirror if applicable.


In [ ]:
# --- Option A: upload code.zip ---
from google.colab import files
import os, zipfile

uploaded = files.upload()  # drag-drop code.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")

def find_code_root(base="/content", max_depth=2):
    for dirpath, _, filenames in os.walk(base):
        depth = dirpath[len(base):].count(os.sep)
        if depth > max_depth:
            continue
        if "requirements.txt" in filenames and os.path.isdir(os.path.join(dirpath, "lora")):
            return dirpath
    return None

root = find_code_root()
assert root is not None, "Couldn't find a folder with requirements.txt + lora/ in the uploaded zip."
os.chdir(root)
print("Working dir:", os.getcwd())
print(os.listdir(root))


In [ ]:
# --- Option B: clone from a public GitHub mirror (uncomment if applicable) ---
# !git clone https://github.com/austinlu2005/myLoRA.git /content/myLoRA
# %cd /content/myLoRA/code


In [ ]:
!pip install -q -r requirements.txt


## 2. Imports


In [ ]:
import sys, os, json, math, random, time
from pathlib import Path

sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import ImageOps
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

from lora.inject import inject_lora
from lora.save_load import load_lora_state_dict, save_lora_state_dict
from training.optim import build_optimizer, build_scheduler
from utils.param_utils import print_trainable_parameters
from utils.seed import set_seed

print(
    "torch:", torch.__version__,
    "| cuda:", torch.cuda.is_available(),
    "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
)


## 3. Hyperparameters

These defaults are chosen for a **notebook-friendly smoke test** rather than a full benchmark run. If you want a stronger run, increase the dataset slice and image size after the pipeline works end-to-end.


In [ ]:
CFG = {
    "checkpoint": "nvidia/mit-b0",
    "dataset_id": "merve/scene_parse_150",
    "dataset_split": "train[:200]",
    "eval_ratio": 0.2,
    "image_size": 256,
    "rank": 8,
    "alpha": 16,
    "dropout": 0.1,
    "lr": 2e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "batch_size": 4,
    "num_epochs": 6,
    "grad_clip": 1.0,
    "seed": 42,
    "log_steps": 10,
    "output_dir": "./runs/segformer_sceneparse150_lora",
}

CFG


## 4. Dataset and Visualization Helpers

`merve/scene_parse_150` is a public subset of ADE20K, so it is convenient for a Colab notebook. SegFormer expects ADE-style label reduction (`0 -> 255 ignore`, everything else shifted down by one), which matches the Hugging Face task guide.


In [ ]:
set_seed(CFG["seed"])

raw = load_dataset(CFG["dataset_id"], split=CFG["dataset_split"]).shuffle(seed=CFG["seed"])
split = raw.train_test_split(test_size=CFG["eval_ratio"], seed=CFG["seed"])
train_raw, eval_raw = split["train"], split["test"]

id2label_path = hf_hub_download(
    repo_id="huggingface/label-files",
    filename="ade20k-id2label.json",
    repo_type="dataset",
)
id2label = json.loads(Path(id2label_path).read_text())
id2label = {int(k): v for k, v in id2label.items()}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)
IGNORE_INDEX = 255

image_processor = AutoImageProcessor.from_pretrained(
    CFG["checkpoint"],
    do_reduce_labels=True,
    size={"height": CFG["image_size"], "width": CFG["image_size"]},
)

print(f"train={len(train_raw)} | eval={len(eval_raw)} | num_labels={num_labels}")
print(train_raw[0])


In [ ]:
def reduce_ade_labels(mask, ignore_index=IGNORE_INDEX):
    mask = np.array(mask, dtype=np.int64)
    mask = mask - 1
    mask[mask == -1] = ignore_index
    return mask

def make_palette(n, seed=0):
    rng = np.random.default_rng(seed)
    palette = rng.integers(0, 255, size=(n, 3), dtype=np.uint8)
    palette[0] = np.array([0, 0, 0], dtype=np.uint8)
    return palette

PALETTE = make_palette(num_labels, seed=0)

def colorize_mask(mask, palette=PALETTE, ignore_index=IGNORE_INDEX):
    mask = np.asarray(mask, dtype=np.int64)
    color = np.zeros((*mask.shape, 3), dtype=np.uint8)
    valid = (mask >= 0) & (mask < len(palette))
    color[valid] = palette[mask[valid]]
    color[mask == ignore_index] = 0
    return color

class SceneParseSegmentationDataset(Dataset):
    def __init__(self, hf_dataset, processor, train=False):
        self.ds = hf_dataset
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        example = self.ds[idx]
        image = example["image"].convert("RGB")
        annotation = example["annotation"]

        if self.train and random.random() < 0.5:
            image = ImageOps.mirror(image)
            annotation = ImageOps.mirror(annotation)

        encoded = self.processor(image, annotation, return_tensors="pt")
        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": encoded["labels"].squeeze(0).long(),
        }

train_ds = SceneParseSegmentationDataset(train_raw, image_processor, train=True)
eval_ds = SceneParseSegmentationDataset(eval_raw, image_processor, train=False)

sample = train_ds[0]
print(sample["pixel_values"].shape, sample["labels"].shape)
print("labels sample:", sample["labels"].unique()[:20])


In [ ]:
def show_raw_examples(hf_ds, n=2):
    fig, axes = plt.subplots(n, 3, figsize=(14, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row in range(n):
        ex = hf_ds[row]
        image = ex["image"].convert("RGB")
        mask = reduce_ade_labels(ex["annotation"])
        colored = colorize_mask(mask)
        overlay = (0.65 * np.array(image) + 0.35 * colored).astype(np.uint8)

        axes[row, 0].imshow(image)
        axes[row, 0].set_title("image")
        axes[row, 1].imshow(colored)
        axes[row, 1].set_title("annotation")
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title("overlay")

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

show_raw_examples(train_raw, n=2)


## 5. Build SegFormer + LoRA

The local `inject_lora(...)` function already works for SegFormer because the attention blocks use `query` and `value` linear projections. After injection, we unfreeze the segmentation `decode_head` so the task head can adapt alongside the LoRA adapters.


In [ ]:
def build_segformer_lora(cfg, id2label, label2id):
    model = AutoModelForSemanticSegmentation.from_pretrained(
        cfg["checkpoint"],
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    model, replaced = inject_lora(
        model,
        target_modules=["query", "value"],
        rank=cfg["rank"],
        alpha=cfg["alpha"],
        dropout=cfg["dropout"],
    )
    for p in model.decode_head.parameters():
        p.requires_grad = True
    return model, replaced

model, replaced = build_segformer_lora(CFG, id2label, label2id)
print(f"Injected LoRA into {len(replaced)} modules")
print(replaced[:8])
print_trainable_parameters(model)


## 6. Segmentation Trainer and Metrics

The repo's generic `Trainer` is aimed at classification / causal LM, so this notebook defines a small segmentation-specific loop here. The model still uses the repo's optimizer and scheduler helpers.


In [ ]:
def fast_hist(preds, labels, num_labels, ignore_index=IGNORE_INDEX):
    mask = labels != ignore_index
    preds = preds[mask].to(torch.int64)
    labels = labels[mask].to(torch.int64)
    if preds.numel() == 0:
        return torch.zeros((num_labels, num_labels), dtype=torch.float64)
    flat = labels * num_labels + preds
    hist = torch.bincount(flat, minlength=num_labels * num_labels)
    return hist.reshape(num_labels, num_labels).to(torch.float64)

def hist_to_metrics(hist):
    intersection = hist.diag()
    union = hist.sum(dim=1) + hist.sum(dim=0) - intersection
    valid = union > 0
    per_class_iou = torch.zeros_like(intersection)
    per_class_iou[valid] = intersection[valid] / union[valid]

    mean_iou = per_class_iou[valid].mean().item() if valid.any() else 0.0
    pixel_accuracy = intersection.sum().item() / hist.sum().item() if hist.sum().item() > 0 else 0.0
    return {
        "mean_iou": mean_iou,
        "pixel_accuracy": pixel_accuracy,
    }

class SegmentationTrainer:
    def __init__(
        self,
        model,
        train_dataset,
        eval_dataset,
        optimizer,
        scheduler,
        batch_size,
        num_epochs,
        device,
        num_labels,
        grad_clip=1.0,
        log_steps=10,
        output_dir=None,
    ):
        self.model = model.to(device)
        self.train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
        self.eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, drop_last=False)
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.device = device
        self.num_labels = num_labels
        self.grad_clip = grad_clip
        self.log_steps = log_steps
        self.output_dir = Path(output_dir) if output_dir else None
        if self.output_dir is not None:
            self.output_dir.mkdir(parents=True, exist_ok=True)

        self.global_step = 0
        self.best_metric = None
        self.history = []

    def _move_to_device(self, batch):
        return {k: v.to(self.device, non_blocking=True) for k, v in batch.items()}

    def train(self):
        for epoch in range(self.num_epochs):
            self._train_one_epoch(epoch)
            metrics = self.evaluate()
            self.history.append({"epoch": epoch, **metrics})
            print(f"[epoch {epoch}] eval: {metrics}")
            self._maybe_save_best(metrics)
        return self.history

    def _train_one_epoch(self, epoch):
        self.model.train()
        running_loss, running_n = 0.0, 0
        pbar = tqdm(self.train_loader, desc=f"epoch {epoch}", leave=False)
        for batch in pbar:
            batch = self._move_to_device(batch)
            out = self.model(**batch)
            loss = out.loss

            self.optimizer.zero_grad()
            loss.backward()
            if self.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(
                    (p for p in self.model.parameters() if p.requires_grad),
                    self.grad_clip,
                )
            self.optimizer.step()
            self.scheduler.step()

            running_loss += loss.item()
            running_n += 1
            self.global_step += 1

            if self.global_step % self.log_steps == 0:
                avg = running_loss / max(running_n, 1)
                pbar.set_postfix(loss=f"{avg:.4f}", lr=f"{self.scheduler.get_last_lr()[0]:.2e}")
                running_loss, running_n = 0.0, 0

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        total_loss, n_batches = 0.0, 0
        hist = torch.zeros((self.num_labels, self.num_labels), dtype=torch.float64)

        for batch in self.eval_loader:
            batch = self._move_to_device(batch)
            out = self.model(**batch)
            total_loss += out.loss.item()
            n_batches += 1

            logits = nn.functional.interpolate(
                out.logits,
                size=batch["labels"].shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            preds = logits.argmax(dim=1).detach().cpu()
            labels = batch["labels"].detach().cpu()
            hist += fast_hist(preds, labels, self.num_labels, ignore_index=IGNORE_INDEX)

        metrics = {"eval_loss": total_loss / max(n_batches, 1)}
        metrics.update(hist_to_metrics(hist))
        return metrics

    def _maybe_save_best(self, metrics):
        if self.output_dir is None:
            return
        score = metrics["mean_iou"]
        if self.best_metric is None or score > self.best_metric:
            self.best_metric = score
            save_lora_state_dict(self.model, self.output_dir / "lora_best.pt")
            torch.save(self.model.decode_head.state_dict(), self.output_dir / "decode_head_best.pt")


## 7. Train


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
steps_per_epoch = math.ceil(len(train_ds) / CFG["batch_size"])
num_training_steps = steps_per_epoch * CFG["num_epochs"]

model, replaced = build_segformer_lora(CFG, id2label, label2id)
optimizer = build_optimizer(model, lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = build_scheduler(
    optimizer,
    num_training_steps=num_training_steps,
    warmup_ratio=CFG["warmup_ratio"],
)

trainer = SegmentationTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    optimizer=optimizer,
    scheduler=scheduler,
    batch_size=CFG["batch_size"],
    num_epochs=CFG["num_epochs"],
    device=device,
    num_labels=num_labels,
    grad_clip=CFG["grad_clip"],
    log_steps=CFG["log_steps"],
    output_dir=CFG["output_dir"],
)

t0 = time.time()
history = trainer.train()
wall = time.time() - t0

result = {
    "config": CFG,
    "train_size": len(train_ds),
    "eval_size": len(eval_ds),
    "replaced_modules": replaced,
    "best_metric": trainer.best_metric,
    "wall_seconds": wall,
    "history": history,
}
output_dir = Path(CFG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)
output_dir.joinpath("result.json").write_text(json.dumps(result, indent=2))

print("best mean IoU:", trainer.best_metric)
print("wall minutes:", round(wall / 60, 2))
print(json.dumps(history[-1], indent=2))
print("saved to", output_dir)


## 8. Reload the Best Checkpoint and Visualize Predictions


In [ ]:
best_model, _ = build_segformer_lora(CFG, id2label, label2id)
load_lora_state_dict(best_model, Path(CFG["output_dir"]) / "lora_best.pt")
best_model.decode_head.load_state_dict(
    torch.load(Path(CFG["output_dir"]) / "decode_head_best.pt", map_location="cpu")
)
best_model = best_model.to(device).eval()

def predict_mask(model, image):
    encoded = image_processor(image, return_tensors="pt")
    pixel_values = encoded["pixel_values"].to(device)
    with torch.no_grad():
        out = model(pixel_values=pixel_values)
    logits = nn.functional.interpolate(
        out.logits,
        size=image.size[::-1],
        mode="bilinear",
        align_corners=False,
    )
    return logits.argmax(dim=1)[0].detach().cpu().numpy()

ex = eval_raw[0]
image = ex["image"].convert("RGB")
true_mask = reduce_ade_labels(ex["annotation"])
pred_mask = predict_mask(best_model, image)

true_color = colorize_mask(true_mask)
pred_color = colorize_mask(pred_mask)
true_overlay = (0.65 * np.array(image) + 0.35 * true_color).astype(np.uint8)
pred_overlay = (0.65 * np.array(image) + 0.35 * pred_color).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(image); axes[0].set_title("image")
axes[1].imshow(true_color); axes[1].set_title("ground truth")
axes[2].imshow(pred_color); axes[2].set_title("prediction")
axes[3].imshow(pred_overlay); axes[3].set_title("prediction overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Notes

- This notebook is an **extension** of the repo's LoRA implementation, not a reproduction of the original 2021 paper.
- For a stronger run, try `CFG["dataset_split"] = "train[:1000]"` or `"train"` and increase `CFG["image_size"]` to `512`.
- If you want to adapt a different segmentation model, inspect its attention module names and pass the appropriate substrings to `inject_lora(...)`.
